# Envelope Strategy — Backtest

A **mean-reversion** strategy: it assumes price tends to snap back toward a moving
average, so it **buys weakness and sells strength**.

**How it works**

1. Compute a moving average (MA) of the close.
2. Draw several **envelope bands** at fixed percentages below (and above) the MA.
3. Each time price pierces a *lower* band, buy a slice of the position
   (**scale-in**) — the further price falls, the bigger the position.
4. Exit the whole position when price reverts back to the central MA.
5. Protective layers: a hard **stop-loss**, an optional **price-jump** gap
   breaker, and an approximated **liquidation** level.

This is *event-driven*: the backtester walks the candles **one bar at a time**,
updating a real `Position` with fees, leverage and stops — closer to live trading
than the vectorized notebooks.

Everything runs on the Polars-native `quant_research` library.

In [ ]:
import polars as pl

from quant_research.connectors import CCXTLoader          # load cached candles
from quant_research.strategies import EnvelopeStrategy     # the strategy logic
from quant_research.backtest import BacktestAnalysis       # metrics + charts

## Step 1 — Load the price data

Reads BTC perpetual 1-hour candles from the cache (run `data_engine.ipynb` first
if the cache is empty).

In [ ]:
loader = CCXTLoader(exchange="binanceusdm")

In [ ]:
symbol = "BTC/USDT:USDT"
# 1-hour candles from 2021 onward, read straight from the local cache.
ohlcv = loader.load(symbol, timeframe="1h", start_date="2021-01-01 00:00:00")

## Step 2 — Configure the strategy

Every knob explained:

| param | meaning |
|-------|---------|
| `average_type` | which moving average to center on: `SMA`, `EMA`, `WMA` or `DCM` (Donchian mid) |
| `average_period` | lookback (in bars) of that MA |
| `envelopes` | band offsets as fractions — `[0.07, 0.11, 0.14]` = bands at ±7%, ±11%, ±14% around the MA |
| `stop_loss_pct` | hard stop distance from average entry — `0.3` = 30% |
| `price_jump_pct` | *(optional)* flat-exit if a bar gaps open past this % (circuit breaker) |
| `position_size_percentage` | % of wallet to deploy, split across the bands |
| `mode` | `'long'`, `'short'` or `'both'` (default `'both'`) |

In [ ]:
strategy_params = {
    "average_type": "SMA",            # centre line: 'SMA', 'EMA', 'WMA', 'DCM'
    "average_period": 6,              # MA lookback, in bars
    "envelopes": [0.07, 0.11, 0.14],  # buy slices 7%, 11%, 14% below the MA (and short above)
    "stop_loss_pct": 0.3,             # bail if price moves 30% against the average entry
    # "price_jump_pct": 0.3,          # optional gap circuit breaker
    "position_size_percentage": 100,  # deploy up to 100% of wallet across the bands
    # "mode": "long" | "short" | "both"  (default 'both')
}

## Step 3 — Run the backtest

`run_backtest` walks the candles bar by bar. Its execution knobs:

- `initial_balance` — starting wallet (quote currency, e.g. 1000 USDT).
- `leverage` — position multiplier; `1` = no leverage (liquidation is far away).
- `open_fee_rate` / `close_fee_rate` — fees per side as fractions. `0.0002` = 2 bps
  (a passive **maker** entry); `0.0006` = 6 bps (an aggressive **taker** exit).

In [ ]:
strategy = EnvelopeStrategy(strategy_params, ohlcv)
strategy.run_backtest(
    initial_balance=1000,     # start with 1000 USDT
    leverage=1,               # no leverage
    open_fee_rate=0.0002,     # 2 bps maker fee on entry
    close_fee_rate=0.0006,    # 6 bps taker fee on exit
)

## Step 4 — Read the performance metrics

`BacktestAnalysis` wraps the finished strategy and computes the usual scorecard:
total return, Sharpe/Sortino/Calmar, max drawdown, win rate, etc.

In [ ]:
results = BacktestAnalysis(strategy)
results.print_metrics()

## Step 5 — Inspect the raw records

- `trades_info` — one row per closed trade (entry/exit, size, P&L, fees).
- `equity_record` — the wallet value bar by bar (the equity curve data).

In [ ]:
strategy.trades_info      # every closed trade

In [ ]:
strategy.equity_record    # wallet value over time

## Step 6 — Visualize

- `plot_equity` — equity curve (optionally over the price).
- `plot_drawdown` — how deep and long the losing stretches were.
- `plot_monthly_performance` — return per calendar month (`year="all"` = every year).

In [ ]:
results.plot_equity()

In [ ]:
results.plot_drawdown()

In [ ]:
results.plot_monthly_performance(year="all")

## Step 7 — Candlestick with the envelope overlaid

`plot_candlestick` draws the price candles and any extra lines we hand it. The
`indicators` dict maps a **name → {color, df}**, where `df` has two columns:
`time` and the value to plot. Below we overlay the central MA plus every upper/lower
band, so you can literally see where each scale-in fired.

In [ ]:
# Overlay the moving average and each envelope band on the price chart.
indicators = {
    "average": {
        "color": "white",
        "df": strategy.data.select(pl.col("datetime").alias("time"), pl.col("average")).drop_nulls(),
    }
}
# Add the high (red) and low (green) band for each envelope level.
for i, _ in enumerate(strategy_params["envelopes"], start=1):
    for side, color in (("high", "red"), ("low", "green")):
        col = f"band_{side}_{i}"
        indicators[col] = {
            "color": color,
            "df": strategy.data.select(pl.col("datetime").alias("time"), pl.col(col)).drop_nulls(),
        }

results.plot_candlestick(indicators=indicators)

## Exercises

1. **Bands** — widen or add envelope levels (e.g. `[0.05, 0.10, 0.15, 0.20]`).
   How does it change trade count and drawdown?
2. **Average** — swap `SMA` for `EMA` or a longer `average_period`.
3. **Costs** — bump both fee rates to a realistic taker/taker (0.0006 / 0.0006).
   Does the edge survive?
4. **Direction** — set `"mode": "long"` to disable shorts and compare.